In [2]:
!pip install torch numpy pandas scikit-learn textblob networkx xgboost joblib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import re
from collections import Counter, defaultdict
import multiprocessing as mp
from functools import partial
import gc
import warnings
from datetime import datetime, timedelta
import networkx as nx
from textblob import TextBlob
import hashlib
import math
import xgboost as xgb
import joblib
import os

warnings.filterwarnings('ignore')

def create_balanced_dataset(json_file_path):
    if not os.path.exists(json_file_path):
        raise FileNotFoundError(f"File not found at specified path: {json_file_path}")
    data = []
    with open(json_file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    for item in data:
        if '_id' in item and isinstance(item['_id'], dict) and '$oid' in item['_id']:
            item['_id'] = item['_id']['$oid']
    df = pd.DataFrame(data)
    unique_classes = df['class'].unique()
    print(f"Available classes: {unique_classes}")
    print(f"Class counts: {df['class'].value_counts()}")
    spam_reviews = df[df['class'] == 0]
    non_spam_reviews = df[df['class'] == 1]
    if len(spam_reviews) == 0:
        raise ValueError("No class 0 reviews found in the dataset. Available classes: " + str(unique_classes.tolist()))
    if len(non_spam_reviews) == 0:
        raise ValueError("No class 1 reviews found in the dataset. Available classes: " + str(unique_classes.tolist()))
    print(f"Class 0 reviews: {len(spam_reviews)}")
    print(f"Class 1 reviews: {len(non_spam_reviews)}")
    if len(spam_reviews) >= 5000:
        spam_sample = spam_reviews.sample(n=5000, random_state=42)
    else:
        print(f"Warning: Only {len(spam_reviews)} class 0 reviews available, sampling with replacement")
        spam_sample = spam_reviews.sample(n=5000, replace=True, random_state=42)
    if len(non_spam_reviews) >= 5000:
        non_spam_sample = non_spam_reviews.sample(n=5000, random_state=42)
    else:
        print(f"Warning: Only {len(non_spam_reviews)} class 1 reviews available, sampling with replacement")
        non_spam_sample = non_spam_reviews.sample(n=5000, replace=True, random_state=42)
    balanced_df = pd.concat([spam_sample, non_spam_sample], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Final balanced dataset:")
    print(f"Total entries: {len(balanced_df)}")
    print(f"Class 0: {sum(balanced_df['class'] == 0)}")
    print(f"Class 1: {sum(balanced_df['class'] == 1)}")
    balanced_dict = balanced_df.to_dict('records')
    return balanced_dict


class FastDataProcessor:
    def __init__(self, json_path, max_total_samples=10000, chunk_size=1000):
        self.json_path = json_path
        self.max_total_samples = max_total_samples
        self.chunk_size = chunk_size
        self.num_workers = min(4, mp.cpu_count())

    def load_data_fast(self):
        all_data=create_balanced_dataset(json_path)
        df = pd.DataFrame(all_data)
        print(f"Loaded {len(df)} samples.")
        return df


class FeatureEngineer:
    def __init__(self, vectorizer_choice='tfidf'):
        self.vectorizer_choice = vectorizer_choice
        self.tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
        self.hashing_vectorizer = HashingVectorizer(n_features=10000, stop_words='english', alternate_sign=False)

    def generate_core_features(self, df):
        print("Generating core features...")
        features_df = pd.DataFrame()
        df['reviewText'] = df['reviewText'].astype(str)
        df['summary'] = df['summary'].astype(str)
        if self.vectorizer_choice == 'tfidf':
            review_text_vectors = self.tfidf_vectorizer.fit_transform(df['reviewText'])
            summary_vectors = self.tfidf_vectorizer.fit_transform(df['summary'])
        else:
            review_text_vectors = self.hashing_vectorizer.fit_transform(df['reviewText'])
            summary_vectors = self.hashing_vectorizer.fit_transform(df['summary'])
        review_text_df = pd.DataFrame(review_text_vectors.toarray(), index=df.index)
        summary_df = pd.DataFrame(summary_vectors.toarray(), index=df.index)
        review_text_df.columns = [f'text_feature_{i}' for i in range(review_text_df.shape[1])]
        summary_df.columns = [f'summary_feature_{i}' for i in range(summary_df.shape[1])]
        features_df = pd.concat([features_df, review_text_df, summary_df], axis=1)
        features_df['overall_rating'] = df['overall']
        features_df['review_length'] = df['reviewText'].apply(len)
        features_df['summary_length'] = df['summary'].apply(len)
        df['reviewTime'] = pd.to_datetime(df['unixReviewTime'], unit='s')
        features_df['day_of_week'] = df['reviewTime'].dt.dayofweek
        features_df['hour_of_day'] = df['reviewTime'].dt.hour
        if 'class' not in df.columns:
            df['class'] = ((df['overall'] <= 2.0) & (df['reviewText'].apply(len) < 50)).astype(int)
        features_df['class'] = df['class']
        features_df['reviewerID'] = df['reviewerID']
        features_df['asin'] = df['asin']
        features_df['unixReviewTime'] = df['unixReviewTime']
        features_df['reviewText'] = df['reviewText']
        features_df['summary'] = df['summary']
        features_df['overall'] = df['overall']
        features_df['reviewTime'] = df['reviewTime']
        print("Core features generated.")
        return features_df


class AdvancedFraudFeatures:
    def __init__(self, df, num_workers=None, max_samples_for_advanced_features=10000, for_inference=False):
        self.df = df
        self.num_workers = num_workers if num_workers is not None else min(4, mp.cpu_count())
        # If for_inference, we don't sample, we use all relevant data passed
        self.max_samples_for_advanced_features = max_samples_for_advanced_features if not for_inference else len(df)
        self.for_inference = for_inference
        # Ensure 'processed_text_for_hash' is only added if not in inference or if it's the full df
        
        # Ensure reviewText is string before processing for hash
        self.df['reviewText'] = self.df['reviewText'].astype(str)
        if not for_inference or 'processed_text_for_hash' not in self.df.columns:
            self.df['processed_text_for_hash'] = self.df['reviewText'].apply(self._preprocess_text_for_hash)

    def _preprocess_text_for_hash(self, text):
        if not isinstance(text, str):
            # Convert non-string types to string; handle NaN by converting to empty string
            text = str(text) if not pd.isna(text) else ""
        text = re.sub(r'[^\w\s]', '', text).lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _get_sampled_df(self):
        if self.for_inference:
            # For inference, use the entire passed df (which should contain relevant history)
            return self.df.copy()
        elif len(self.df) > self.max_samples_for_advanced_features:
            print(f"Sampling {self.max_samples_for_advanced_features} for advanced feature calculation...")
            return self.df.sample(n=self.max_samples_for_advanced_features, random_state=42).reset_index(drop=True)
        return self.df.copy()

    def detect_coordinated_attacks(self, df):
        sampled_df = self._get_sampled_df()
        print("Detecting coordinated attacks (product-centric)...")
        coordinated_attack_features = defaultdict(
            lambda: {'num_coordinated_reviews_on_product': 0, 'avg_time_diff_on_product': 0, 'rating_variance_on_product': 0}
        )
        sampled_df['reviewTime_dt'] = pd.to_datetime(sampled_df['unixReviewTime'], unit='s', errors='coerce')
        sampled_df = sampled_df.dropna(subset=['reviewTime_dt'])

        product_groups = sampled_df.groupby('asin')

        for asin, product_df in product_groups:
            if len(product_df) > 1:
                # Sort reviews for the product by time
                product_df = product_df.sort_values(by='reviewTime_dt').reset_index(drop=True)

                time_diffs_seconds = product_df['reviewTime_dt'].diff().dt.total_seconds().dropna()
                avg_time_diff = time_diffs_seconds.mean() if not time_diffs_seconds.empty else 0
                rating_variance = product_df['overall'].var() if len(product_df['overall']) > 1 else 0

                num_coordinated_reviews = len(product_df)

                for _, row in product_df.iterrows():
                    reviewer_id = row['reviewerID']
                    # Aggregate features for each reviewer, sum or average across products
                    coordinated_attack_features[reviewer_id]['num_coordinated_reviews_on_product'] += num_coordinated_reviews
                    coordinated_attack_features[reviewer_id]['avg_time_diff_on_product'] += avg_time_diff
                    coordinated_attack_features[reviewer_id]['rating_variance_on_product'] += rating_variance

        # Normalize the aggregated features by the number of products a reviewer was involved in
        for reviewer_id, features in coordinated_attack_features.items():
            product_count = sampled_df[sampled_df['reviewerID'] == reviewer_id]['asin'].nunique()
            if product_count > 0:
                features['num_coordinated_reviews_on_product'] /= product_count
                features['avg_time_diff_on_product'] /= product_count
                features['rating_variance_on_product'] /= product_count
        
        print(f"Coordinated attack detection (product-centric) complete for {len(coordinated_attack_features)} reviewers.")
        return coordinated_attack_features

    def detect_template_reviews(self, df):
        sampled_df = self._get_sampled_df()
        print("Detecting template reviews using hashing (optimized)...")
        template_flags = defaultdict(lambda: {'template_review_similarity_score': 0, 'template_review_count': 0})
        reviews_by_hash = defaultdict(list)
        for idx, row in sampled_df.iterrows():
            # Explicitly check if 'processed_text_for_hash' is a string and not empty
            if isinstance(row['processed_text_for_hash'], str) and row['processed_text_for_hash']:
                text_hash = hashlib.md5(row['processed_text_for_hash'].encode('utf-8')).hexdigest()
                reviews_by_hash[text_hash].append({'reviewerID': row['reviewerID'], 'text': row['reviewText']})
        for text_hash, review_list in reviews_by_hash.items():
            if len(review_list) > 1:
                for review_info in review_list:
                    reviewer_id = review_info['reviewerID']
                    template_flags[reviewer_id]['template_review_similarity_score'] = 1.0
                    template_flags[reviewer_id]['template_review_count'] = len(review_list) - 1
        print(f"Identified {len([h for h, r_list in reviews_by_hash.items() if len(r_list) > 1])} groups of identical template reviews.")
        return template_flags

    def analyze_linguistic_fingerprints(self, df):
        sampled_df = self._get_sampled_df()
        print("Analyzing linguistic fingerprints...")
        linguistic_features = defaultdict(lambda: {'avg_word_length': 0, 'sentiment_polarity': 0,
                                                   'sentiment_subjectivity': 0, 'vocab_richness': 0,
                                                   'sentence_consistency': 0})
        _process_review_partial = partial(self._process_single_review_linguistic, df=sampled_df)
        with mp.Pool(processes=self.num_workers) as pool:
            results = pool.map(_process_review_partial, sampled_df.index.tolist())
        for res in results:
            if res:
                reviewer_id = res['reviewerID']
                linguistic_features[reviewer_id]['avg_word_length'] = res['avg_word_length']
                linguistic_features[reviewer_id]['sentiment_polarity'] = res['sentiment_polarity']
                linguistic_features[reviewer_id]['sentiment_subjectivity'] = res['sentiment_subjectivity']
                linguistic_features[reviewer_id]['vocab_richness'] = res['vocab_richness']
                linguistic_features[reviewer_id]['sentence_consistency'] = res['sentence_consistency']
        return linguistic_features

    def _process_single_review_linguistic(self, idx, df):
        row = df.loc[idx]
        text = row['reviewText']
        if not isinstance(text, str) or not text.strip():
            return None
        try:
            blob = TextBlob(text)
            words = [word for word in blob.words if word.isalpha()]
            avg_word_length = sum(len(word) for word in words) / len(words) if words else 0
            sentiment_polarity = blob.sentiment.polarity
            sentiment_subjectivity = blob.sentiment.subjectivity
            vocab_richness = len(set(words)) / len(words) if words else 0
            sentences = [s.strip() for s in blob.sentences if s.strip()]
            sentence_lengths = [len(s.split()) for s in sentences]
            sentence_consistency = np.var(sentence_lengths) if len(sentence_lengths) > 1 else 0
            return {
                'reviewerID': row['reviewerID'],
                'avg_word_length': avg_word_length,
                'sentiment_polarity': sentiment_polarity,
                'sentiment_subjectivity': sentiment_subjectivity,
                'vocab_richness': vocab_richness,
                'sentence_consistency': sentence_consistency
            }
        except Exception:
            return None

    def analyze_emotional_manipulation(self, df):
        sampled_df = self._get_sampled_df()
        print("Analyzing emotional manipulation...")
        emotional_flags = defaultdict(lambda: {'extreme_sentiment_count': 0, 'exclamation_ratio': 0, 'question_ratio': 0, 'all_caps_ratio': 0})
        _process_review_partial = partial(self._process_single_review_emotional, df=sampled_df)
        with mp.Pool(processes=self.num_workers) as pool:
            results = pool.map(_process_review_partial, sampled_df.index.tolist())
        for res in results:
            if res:
                reviewer_id = res['reviewerID']
                emotional_flags[reviewer_id]['extreme_sentiment_count'] += res['extreme_sentiment_count']
                emotional_flags[reviewer_id]['exclamation_ratio'] += res['exclamation_ratio']
                emotional_flags[reviewer_id]['question_ratio'] += res['question_ratio']
                emotional_flags[reviewer_id]['all_caps_ratio'] += res['all_caps_ratio']
        return emotional_flags

    def _process_single_review_emotional(self, idx, df):
        row = df.loc[idx]
        text = row['reviewText']
        if not isinstance(text, str) or not text.strip():
            return None
        extreme_sentiment_count = 0
        exclamation_ratio = 0
        question_ratio = 0
        all_caps_ratio = 0
        try:
            blob = TextBlob(text)
            if abs(blob.sentiment.polarity) > 0.8:
                extreme_sentiment_count = 1
        except Exception:
            pass
        total_chars = len(text)
        if total_chars > 0:
            exclamation_ratio = text.count('!') / total_chars
            question_ratio = text.count('?') / total_chars
            all_caps_chars = sum(1 for char in text if char.isupper() and char.isalpha())
            all_caps_ratio = all_caps_chars / total_chars
        return {
            'reviewerID': row['reviewerID'],
            'extreme_sentiment_count': extreme_sentiment_count,
            'exclamation_ratio': exclamation_ratio,
            'question_ratio': question_ratio,
            'all_caps_ratio': all_caps_ratio
        }

    def detect_timing_anomalies(self, df):
        sampled_df = self._get_sampled_df()
        print("Detecting timing anomalies...")
        timing_anomalies = defaultdict(lambda: {'avg_review_interval_days': 0, 'review_burstiness_score': 0})
        sampled_df['reviewTime_dt'] = pd.to_datetime(sampled_df['unixReviewTime'], unit='s', errors='coerce')
        sampled_df = sampled_df.dropna(subset=['reviewTime_dt'])
        reviewer_groups = sampled_df.sort_values(by='reviewTime_dt').groupby('reviewerID')
        for reviewer_id, group_df in reviewer_groups:
            if len(group_df) > 1:
                time_diffs_seconds = group_df['reviewTime_dt'].diff().dt.total_seconds().dropna()
                if not time_diffs_seconds.empty:
                    avg_interval_seconds = time_diffs_seconds.mean()
                    avg_review_interval_days = avg_interval_seconds / (60 * 60 * 24)
                    burstiness_score = np.std(time_diffs_seconds) / avg_interval_seconds if avg_interval_seconds > 0 else 0
                    timing_anomalies[reviewer_id]['avg_review_interval_days'] = avg_review_interval_days
                    timing_anomalies[reviewer_id]['review_burstiness_score'] = burstiness_score
        return timing_anomalies

    def sliding_window_temporal_analysis(self, df, window_size_days=7, min_reviews_in_window=3):
        sampled_df = self._get_sampled_df()
        print(f"Performing sliding window temporal analysis with window size {window_size_days} days...")
        sampled_df['reviewTime_dt'] = pd.to_datetime(sampled_df['unixReviewTime'], unit='s', errors='coerce')
        sampled_df = sampled_df.dropna(subset=['reviewTime_dt']).sort_values(by='reviewTime_dt').reset_index(drop=True)

        reviewer_burst_flags = defaultdict(lambda: {'burst_review_count': 0, 'consecutive_burst_windows': 0, 'max_burst_reviews_in_window': 0})
        
        unique_reviewers = sampled_df['reviewerID'].unique()
        for reviewer_id in unique_reviewers:
            reviewer_df = sampled_df[sampled_df['reviewerID'] == reviewer_id].sort_values(by='reviewTime_dt')
            if len(reviewer_df) < min_reviews_in_window:
                continue

            reviewer_review_times = reviewer_df['reviewTime_dt'].tolist()
            
            burst_count = 0
            consecutive_bursts = 0
            max_burst_reviews = 0

            for i in range(len(reviewer_review_times)):
                window_start = reviewer_review_times[i]
                window_end = window_start + timedelta(days=window_size_days)
                
                reviews_in_window = [
                    t for t in reviewer_review_times 
                    if t >= window_start and t < window_end
                ]
                
                if len(reviews_in_window) >= min_reviews_in_window:
                    burst_count += 1
                    max_burst_reviews = max(max_burst_reviews, len(reviews_in_window))
                    
                    if i > 0 and (window_start - reviewer_review_times[i-1]).days < window_size_days:
                        consecutive_bursts += 1
                    else:
                        consecutive_bursts = 1 
            
            reviewer_burst_flags[reviewer_id]['burst_review_count'] = burst_count
            reviewer_burst_flags[reviewer_id]['consecutive_burst_windows'] = consecutive_bursts
            reviewer_burst_flags[reviewer_id]['max_burst_reviews_in_window'] = max_burst_reviews
        
        print("Sliding window temporal analysis complete.")
        return reviewer_burst_flags


class LSHSearcher:
    def __init__(self, num_dimensions, num_bands=10, hashes_per_band=5, seed=42):
        self.num_bands = num_bands
        self.hashes_per_band = hashes_per_band
        self.num_hash_functions = num_bands * hashes_per_band
        self.rng = np.random.default_rng(seed)
        
        # Generate random hyperplanes for random projection LSH
        self.random_hyperplanes = self.rng.normal(0, 1, (self.num_hash_functions, num_dimensions))
        
        # Stores reviewer IDs by hash bucket
        self.hash_tables = [defaultdict(list) for _ in range(num_bands)]
        # Stores original signature vectors for re-ranking
        self.signatures = {}

    def _hash_vector(self, vector):
        # Project vector onto random hyperplanes
        projections = np.dot(self.random_hyperplanes, vector)
        # Convert to binary hash: 1 if projection > 0, 0 otherwise
        binary_hashes = (projections > 0).astype(int)
        
        band_hashes = []
        for i in range(self.num_bands):
            start_idx = i * self.hashes_per_band
            end_idx = start_idx + self.hashes_per_band
            band_hash_bits = binary_hashes[start_idx:end_idx]
            # Convert binary hash to a string or tuple for dictionary key
            band_hashes.append(tuple(band_hash_bits))
        return band_hashes

    def index(self, reviewer_id, signature_vector):
        if reviewer_id in self.signatures:
            return # Already indexed
        
        self.signatures[reviewer_id] = signature_vector
        band_hashes = self._hash_vector(signature_vector)
        
        for i, band_hash in enumerate(band_hashes):
            self.hash_tables[i][band_hash].append(reviewer_id)

    def query(self, query_signature_vector, num_results=5):
        candidate_reviewers = set()
        query_band_hashes = self._hash_vector(query_signature_vector)
        
        for i, band_hash in enumerate(query_band_hashes):
            candidates_in_bucket = self.hash_tables[i].get(band_hash, [])
            candidate_reviewers.update(candidates_in_bucket)
            
        # Remove the query reviewer itself if present
        if query_signature_vector is not None: # check if signature exists
            # This is a bit tricky: `query_signature_vector` might not have a direct reviewerID associated if it's a new one.
            # We would typically pass the reviewer_id if it's an existing one we're querying.
            # Assuming for now `query_signature_vector`'s reviewer_id is not in candidates.
            pass # No direct way to remove by signature content.
        
        # If no candidates found, return empty
        if not candidate_reviewers:
            return []

        # Re-rank candidates by actual cosine similarity
        similarities = []
        for reviewer_id in candidate_reviewers:
            if reviewer_id not in self.signatures:
                continue # Should not happen if indexed correctly
            candidate_vector = self.signatures[reviewer_id]
            
            # Calculate cosine similarity
            similarity = np.dot(query_signature_vector, candidate_vector) / \
                         (np.linalg.norm(query_signature_vector) * np.linalg.norm(candidate_vector))
            similarities.append((similarity, reviewer_id))
            
        # Sort by similarity in descending order and return top N
        similarities.sort(key=lambda x: x[0], reverse=True)
        return [(rev_id, sim) for sim, rev_id in similarities[:num_results]]


class HybridFraudSystem:
    def __init__(self):
        self.feature_engineer = FeatureEngineer()
        self.model = None
        self.scaler = None
        self.cached_df = None
        self._feature_names_in_order = None
        self.review_fraud_probabilities = pd.DataFrame()
        self.lsh_searcher = None
        self.reviewer_behavioral_signatures = {} # Stores reviewer_id -> signature vector
        
        self.behavioral_feature_columns = [
            'fraud_graph_num_coordinated_reviews_on_product',
            'fraud_graph_avg_time_diff_on_product',
            'fraud_graph_rating_variance_on_product',
            'fraud_linguistic_avg_word_length',
            'fraud_linguistic_sentiment_polarity',
            'fraud_linguistic_sentiment_subjectivity',
            'fraud_linguistic_vocab_richness',
            'fraud_linguistic_sentence_consistency',
            'fraud_template_template_review_similarity_score',
            'fraud_template_template_review_count',
            'fraud_emotional_extreme_sentiment_count',
            'fraud_emotional_exclamation_ratio',
            'fraud_emotional_question_ratio',
            'fraud_emotional_all_caps_ratio',
            'fraud_timing_avg_review_interval_days',
            'fraud_timing_review_burstiness_score',
            'fraud_sliding_window_burst_review_count',
            'fraud_sliding_window_consecutive_burst_windows',
            'fraud_sliding_window_max_burst_reviews_in_window'
        ]

    def train(self, json_path, force_retrain=False):
        print("Starting HybridFraudSystem training (with XGBoost)...")
        data_processor = FastDataProcessor(json_path)
        df = data_processor.load_data_fast()
        self.cached_df = df
        if df.empty:
            print("No data loaded. Exiting training.")
            return
        df['overall'] = pd.to_numeric(df['overall'], errors='coerce').fillna(0)
        df['unixReviewTime'] = pd.to_numeric(df['unixReviewTime'], errors='coerce').fillna(0)
        print("Generating core features...")
        core_features_df = self.feature_engineer.generate_core_features(df)
        print("Generating advanced fraud features...")
        # For training, AdvancedFraudFeatures operates on the full dataset
        advanced_features_analyzer = AdvancedFraudFeatures(df, max_samples_for_advanced_features=10000)
        coordinated_attacks = advanced_features_analyzer.detect_coordinated_attacks(df)
        linguistic_fingerprints = advanced_features_analyzer.analyze_linguistic_fingerprints(df)
        template_reviews = advanced_features_analyzer.detect_template_reviews(df)
        emotional_manipulation = advanced_features_analyzer.analyze_emotional_manipulation(df)
        timing_anomalies = advanced_features_analyzer.detect_timing_anomalies(df)
        
        sliding_window_features = advanced_features_analyzer.sliding_window_temporal_analysis(df)

        coordinated_attacks_df = pd.DataFrame.from_dict(coordinated_attacks, orient='index').add_prefix('fraud_graph_')
        linguistic_fingerprints_df = pd.DataFrame.from_dict(linguistic_fingerprints, orient='index').add_prefix('fraud_linguistic_')
        template_reviews_df = pd.DataFrame.from_dict(template_reviews, orient='index').add_prefix('fraud_template_')
        emotional_manipulation_df = pd.DataFrame.from_dict(emotional_manipulation, orient='index').add_prefix('fraud_emotional_')
        timing_anomalies_df = pd.DataFrame.from_dict(timing_anomalies, orient='index').add_prefix('fraud_timing_')
        sliding_window_df = pd.DataFrame.from_dict(sliding_window_features, orient='index').add_prefix('fraud_sliding_window_')

        combined_df = core_features_df.set_index('reviewerID', drop=False)
        combined_df = combined_df.merge(coordinated_attacks_df, left_index=True, right_index=True, how='left')
        combined_df = combined_df.merge(linguistic_fingerprints_df, left_index=True, right_index=True, how='left')
        combined_df = combined_df.merge(template_reviews_df, left_index=True, right_index=True, how='left')
        combined_df = combined_df.merge(emotional_manipulation_df, left_index=True, right_index=True, how='left')
        combined_df = combined_df.merge(timing_anomalies_df, left_index=True, right_index=True, how='left')
        combined_df = combined_df.merge(sliding_window_df, left_index=True, right_index=True, how='left')
        combined_df = combined_df.fillna(0)
        print(f"Combined features shape: {combined_df.shape}")

        # Store the full combined_df (with all features) for later use, especially for behavioral fingerprinting
        self.full_feature_df = combined_df.copy() # This will hold all calculated features for all reviewers

        # --- Behavioral Fingerprinting ---
        print("Generating behavioral signatures and indexing with LSH...")
        unique_reviewers_df = combined_df.drop_duplicates(subset=['reviewerID'])
        
        # Ensure all behavioral feature columns exist before selection
        for col in self.behavioral_feature_columns:
            if col not in unique_reviewers_df.columns:
                unique_reviewers_df[col] = 0.0 # Add missing columns with default value
        
        behavioral_features_df = unique_reviewers_df[self.behavioral_feature_columns + ['reviewerID']]
        
        # Handle cases where all selected behavioral features are zero/NaN for a reviewer
        behavioral_features_df = behavioral_features_df.replace([np.inf, -np.inf], np.nan).fillna(0)

        # Scale behavioral features for LSH consistency (optional but good practice)
        behavioral_scaler = StandardScaler()
        behavioral_signatures_scaled = behavioral_scaler.fit_transform(behavioral_features_df[self.behavioral_feature_columns])

        self.lsh_searcher = LSHSearcher(num_dimensions=len(self.behavioral_feature_columns))
        self.reviewer_behavioral_signatures = {}

        for idx, row in behavioral_features_df.iterrows():
            reviewer_id = row['reviewerID']
            # Get the scaled signature for the current reviewer
            signature_vector = behavioral_signatures_scaled[behavioral_features_df.index.get_loc(idx)]
            self.lsh_searcher.index(reviewer_id, signature_vector)
            self.reviewer_behavioral_signatures[reviewer_id] = signature_vector # Store for later querying

        print("LSH indexing complete.")
        # --- End Behavioral Fingerprinting ---


        X = combined_df.drop(['class', 'reviewerID', 'reviewText', 'summary', 'asin', 'unixReviewTime', 'reviewTime', 'overall', 'processed_text_for_hash'], axis=1, errors='ignore')
        y = combined_df['class']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        self.scaler = scaler
        self._feature_names_in_order = X.columns.tolist()
        print("Training XGBoost model...")
        self.model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)
        self.model.fit(X_train_scaled, y_train)
        print("Evaluating XGBoost model on test set...")
        y_pred = self.model.predict(X_test_scaled)
        y_pred_proba = self.model.predict_proba(X_test_scaled)[:, 1]
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred, target_names=['legitimate', 'fraudulent'], zero_division=0))
        joblib.dump(self.model, 'xgboost_fraud_detection_model.joblib')
        joblib.dump(self.scaler, 'scaler.joblib')
        with open('feature_names.json', 'w') as f:
            json.dump(self._feature_names_in_order, f)
        print("XGBoost model, scaler, and feature names saved.")
        X_all = combined_df.drop(['class', 'reviewerID', 'reviewText', 'summary', 'asin', 'unixReviewTime', 'reviewTime', 'overall', 'processed_text_for_hash'], axis=1, errors='ignore')
        X_all_scaled = self.scaler.transform(X_all)
        all_review_probabilities = self.model.predict_proba(X_all_scaled)[:, 1]
        self.review_fraud_probabilities = pd.DataFrame({
            'reviewerID': combined_df['reviewerID'],
            'asin': combined_df['asin'],
            'fraud_probability': all_review_probabilities,
            'is_fraudulent': (all_review_probabilities > 0.5).astype(int)
        })
        print("Review-level fraud probabilities stored for seller/product analysis.")

    def classify_review(self, review_text, summary, overall, reviewer_id, asin):
        single_review_data = {
            'reviewText': review_text,
            'summary': summary,
            'overall': overall,
            'reviewerID': reviewer_id,
            'asin': asin,
            'unixReviewTime': datetime.now().timestamp(),
            'reviewTime': datetime.now(),
            'class': 0 # Placeholder, actual class is unknown for new review
        }
        single_review_df = pd.DataFrame([single_review_data])

        # Combine new review with relevant historical data for incremental feature calculation
        combined_inference_df = single_review_df.copy()
        if self.cached_df is not None:
            # Get historical reviews for this reviewer and product
            historical_reviewer_reviews = self.cached_df[self.cached_df['reviewerID'] == reviewer_id]
            historical_product_reviews = self.cached_df[self.cached_df['asin'] == asin]

            # Concatenate unique historical reviews, ensuring no duplicates with the new review
            # For simplicity, we assume the new review has a unique 'unixReviewTime' or we handle duplicates by keeping the new one
            combined_inference_df = pd.concat([
                historical_reviewer_reviews,
                historical_product_reviews,
                single_review_df
            ]).drop_duplicates(subset=['reviewerID', 'asin', 'unixReviewTime']).reset_index(drop=True)
            
            # Ensure the new review is at the end if we were to sort chronologically for any feature
            combined_inference_df = combined_inference_df.sort_values(by='unixReviewTime').reset_index(drop=True)


        # Generate core features on the combined (new + historical) data
        # FeatureEngineer needs to be fit on a larger corpus (during train), here it just transforms
        # Re-initialize feature engineer for inference to ensure vectorizers are loaded/fitted correctly
        self.feature_engineer = FeatureEngineer() # This re-initializes vectorizers. For proper streaming, they should be pre-fitted.
                                                # In a real streaming system, vectorizers would be pre-trained and loaded.
        core_features_combined = self.feature_engineer.generate_core_features(combined_inference_df)


        # Initialize AdvancedFraudFeatures for inference with the combined DataFrame, ensuring no sampling
        advanced_features_analyzer_inference = AdvancedFraudFeatures(combined_inference_df, for_inference=True)

        coordinated_attacks = advanced_features_analyzer_inference.detect_coordinated_attacks(combined_inference_df)
        linguistic_fingerprints = advanced_features_analyzer_inference.analyze_linguistic_fingerprints(combined_inference_df)
        template_reviews = advanced_features_analyzer_inference.detect_template_reviews(combined_inference_df)
        emotional_manipulation = advanced_features_analyzer_inference.analyze_emotional_manipulation(combined_inference_df)
        timing_anomalies = advanced_features_analyzer_inference.detect_timing_anomalies(combined_inference_df)
        sliding_window_features = advanced_features_analyzer_inference.sliding_window_temporal_analysis(combined_inference_df)

        coordinated_attacks_df = pd.DataFrame.from_dict(coordinated_attacks, orient='index').add_prefix('fraud_graph_')
        linguistic_fingerprints_df = pd.DataFrame.from_dict(linguistic_fingerprints, orient='index').add_prefix('fraud_linguistic_')
        template_reviews_df = pd.DataFrame.from_dict(template_reviews, orient='index').add_prefix('fraud_template_')
        emotional_manipulation_df = pd.DataFrame.from_dict(emotional_manipulation, orient='index').add_prefix('fraud_emotional_')
        timing_anomalies_df = pd.DataFrame.from_dict(timing_anomalies, orient='index').add_prefix('fraud_timing_')
        sliding_window_df = pd.DataFrame.from_dict(sliding_window_features, orient='index').add_prefix('fraud_sliding_window_')

        # Combine features. Use the 'reviewerID' of the single new review to get its specific features.
        # Ensure we only pick the row corresponding to the specific review being classified.
        # This requires matching on reviewerID and potentially reviewTime or other unique identifiers
        # For simplicity, assuming the last row of `core_features_combined` corresponds to the new review
        # if the combined_inference_df was sorted and the new review was appended at the end.
        
        # A more robust way: use the original single_review_df's index or unique identifier
        single_review_core_features = core_features_combined[
            (core_features_combined['reviewerID'] == reviewer_id) & 
            (core_features_combined['unixReviewTime'] == single_review_data['unixReviewTime'])
        ].copy() # Ensure we get the exact row for the new review

        if single_review_core_features.empty:
            # Fallback if the specific new review couldn't be located (e.g., due to duplicate time/reviewer/asin)
            print("Warning: Could not precisely locate the new review within combined features. Using the last row.")
            single_review_core_features = core_features_combined.iloc[[-1]].copy()

        # Merge advanced features for the specific reviewer of the new review
        single_review_features = single_review_core_features.set_index('reviewerID', drop=False)
        single_review_features = single_review_features.merge(coordinated_attacks_df, left_index=True, right_index=True, how='left')
        single_review_features = single_review_features.merge(linguistic_fingerprints_df, left_index=True, right_index=True, how='left')
        single_review_features = single_review_features.merge(template_reviews_df, left_index=True, right_index=True, how='left')
        single_review_features = single_review_features.merge(emotional_manipulation_df, left_index=True, right_index=True, how='left')
        single_review_features = single_review_features.merge(timing_anomalies_df, left_index=True, right_index=True, how='left')
        single_review_features = single_review_features.merge(sliding_window_df, left_index=True, right_index=True, how='left')
        single_review_features = single_review_features.fillna(0)

        if self.scaler is None or self.model is None or self._feature_names_in_order is None:
            try:
                self.scaler = joblib.load('scaler.joblib')
                self.model = joblib.load('xgboost_fraud_detection_model.joblib')
                with open('feature_names.json', 'r') as f:
                    self._feature_names_in_order = json.load(f)
            except FileNotFoundError:
                raise RuntimeError("XGBoost model, scaler, or feature names not found. Please train the system first.")
        
        # Align features for inference
        X_inference_aligned = pd.DataFrame(0.0, index=[0], columns=self._feature_names_in_order)
        for col in self._feature_names_in_order:
            if col in single_review_features.columns:
                X_inference_aligned[col] = single_review_features[col].iloc[0]
        
        X_scaled = self.scaler.transform(X_inference_aligned)
        prediction_proba = self.model.predict_proba(X_scaled)[:, 1]
        probability_fraudulent = prediction_proba.item()
        classification = "fraudulent" if probability_fraudulent > 0.5 else "legitimate"
        flags = {}
        if not coordinated_attacks_df.empty and 'fraud_graph_num_coordinated_reviews_on_product' in coordinated_attacks_df.columns:
            if coordinated_attacks_df['fraud_graph_num_coordinated_reviews_on_product'].get(reviewer_id, 0) > 1: # Get for current reviewer
                flags['coordinated_attack_on_product'] = True
        if not template_reviews_df.empty and 'fraud_template_review_count' in template_reviews_df.columns:
            if template_reviews_df['fraud_template_review_count'].get(reviewer_id, 0) > 0: # Get for current reviewer
                flags['template_review_detected'] = True
        if not emotional_manipulation_df.empty and 'fraud_emotional_extreme_sentiment_count' in emotional_manipulation_df.columns:
            if emotional_manipulation_df['fraud_emotional_extreme_sentiment_count'].get(reviewer_id, 0) > 0: # Get for current reviewer
                flags['extreme_sentiment_detected'] = True
        if not sliding_window_df.empty and 'fraud_sliding_window_max_burst_reviews_in_window' in sliding_window_df.columns:
            if sliding_window_df['fraud_sliding_window_max_burst_reviews_in_window'].get(reviewer_id, 0) > 0: # Get for current reviewer
                flags['review_burst_detected'] = True
        return {
            'classification': classification,
            'confidence': probability_fraudulent if classification == 'fraudulent' else (1 - probability_fraudulent),
            'flags': flags,
            'class_probabilities': {'legitimate': 1 - probability_fraudulent, 'fraudulent': probability_fraudulent}
        }

    def classify_seller_or_product(self, target_id, id_type='asin'):
        if self.review_fraud_probabilities.empty:
            raise RuntimeError("Review fraud probabilities are not available. Please train the system first.")
        if id_type not in ['asin', 'reviewerID']:
            raise ValueError("id_type must be 'asin' or 'reviewerID'.")
        if id_type == 'asin':
            relevant_reviews = self.review_fraud_probabilities[self.review_fraud_probabilities['asin'] == target_id]
        else:
            relevant_reviews = self.review_fraud_probabilities[self.review_fraud_probabilities['reviewerID'] == target_id]
        if relevant_reviews.empty:
            return {
                'id_type': id_type,
                'id': target_id,
                'classification': 'no_data',
                'confidence': 0.0,
                'details': 'No reviews found for this ID.'
            }
        total_reviews = len(relevant_reviews)
        num_fraudulent_reviews = relevant_reviews['is_fraudulent'].sum()
        avg_fraud_probability = relevant_reviews['fraud_probability'].mean()
        if num_fraudulent_reviews > 0 and avg_fraud_probability > 0.7:
            classification = 'highly_suspicious'
            confidence = avg_fraud_probability
        elif num_fraudulent_reviews > 0 and avg_fraud_probability > 0.4:
            classification = 'suspicious'
            confidence = avg_fraud_probability
        else:
            classification = 'legitimate'
            confidence = 1 - avg_fraud_probability
        details = {
            'total_reviews': total_reviews,
            'num_fraudulent_reviews': num_fraudulent_reviews,
            'percentage_fraudulent': (num_fraudulent_reviews / total_reviews) * 100,
            'average_fraud_probability_per_review': avg_fraud_probability
        }
        if self.cached_df is not None:
            if id_type == 'asin':
                original_relevant_data = self.cached_df[self.cached_df['asin'] == target_id]
            else:
                original_relevant_data = self.cached_df[self.cached_df['reviewerID'] == target_id]
            if not original_relevant_data.empty:
                if id_type == 'asin':
                    details['num_unique_reviewers'] = original_relevant_data['reviewerID'].nunique()
        return {
            'id_type': id_type,
            'id': target_id,
            'classification': classification,
            'confidence': confidence,
            'details': details
        }

    def find_similar_reviewers_by_behavior(self, reviewer_id, num_results=5):
        if self.lsh_searcher is None or not self.reviewer_behavioral_signatures:
            raise RuntimeError("LSH system not initialized. Please train the system first.")

        if reviewer_id not in self.reviewer_behavioral_signatures:
            # If the reviewer_id is new or not in the trained set,
            # we need to calculate their behavioral signature first.
            # This would typically involve processing their historical reviews (if any)
            # and the current review to generate a full signature.
            # For this example, we'll assume we are querying for an *existing* reviewer.
            # If a new reviewer's signature needs to be queried, a full pipeline
            # to compute their behavioral features on demand would be needed.
            return {"reviewer_id": reviewer_id, "similar_reviewers": [], "message": "Reviewer ID not found in behavioral signature database."}
        
        query_signature = self.reviewer_behavioral_signatures[reviewer_id]
        
        # Query LSH for similar signatures, excluding the query reviewer itself (handled by LSHSearcher internally logic)
        similar_reviewer_ids_with_scores = self.lsh_searcher.query(query_signature, num_results=num_results+1) # +1 to potentially remove self
        
        # Filter out the query reviewer if it's in the results
        filtered_results = []
        for sim_rev_id, score in similar_reviewer_ids_with_scores:
            if sim_rev_id != reviewer_id:
                filtered_results.append((sim_rev_id, score))
            if len(filtered_results) >= num_results:
                break

        return {
            "reviewer_id": reviewer_id,
            "similar_reviewers": [{"reviewerID": rid, "similarity_score": float(score)} for rid, score in filtered_results]
        }

    def comprehensive_fraud_analysis(self, reviews_df):
        print("Running comprehensive fraud analysis...")
        report_advanced_features_analyzer = AdvancedFraudFeatures(reviews_df, max_samples_for_advanced_features=min(len(reviews_df), 20000))
        coordinated_attacks = report_advanced_features_analyzer.detect_coordinated_attacks(reviews_df)
        template_reviews = report_advanced_features_analyzer.detect_template_reviews(reviews_df)
        timing_anomalies = report_advanced_features_analyzer.detect_timing_anomalies(reviews_df)
        sliding_window_features = report_advanced_features_analyzer.sliding_window_temporal_analysis(reviews_df)

        coordinated_attacks_df = pd.DataFrame.from_dict(coordinated_attacks, orient='index')
        template_reviews_df = pd.DataFrame.from_dict(template_reviews, orient='index')
        timing_anomalies_df = pd.DataFrame.from_dict(timing_anomalies, orient='index')
        sliding_window_features_df = pd.DataFrame.from_dict(sliding_window_features, orient='index')

        num_coordinated_reviews = len(coordinated_attacks_df[coordinated_attacks_df['num_coordinated_reviews_on_product'] > 1]) if not coordinated_attacks_df.empty and 'num_coordinated_reviews_on_product' in coordinated_attacks_df.columns else 0
        avg_time_diff_on_product = coordinated_attacks_df['avg_time_diff_on_product'].mean() if not coordinated_attacks_df.empty and 'avg_time_diff_on_product' in coordinated_attacks_df.columns else 0
        avg_rating_variance_on_product = coordinated_attacks_df['rating_variance_on_product'].mean() if not coordinated_attacks_df.empty and 'rating_variance_on_product' in coordinated_attacks_df.columns else 0
        num_template_reviews = len(template_reviews_df[template_reviews_df['template_review_count'] > 0]) if not template_reviews_df.empty and 'template_review_count' in template_reviews_df.columns else 0
        avg_burstiness = timing_anomalies_df['review_burstiness_score'].mean() if 'review_burstiness_score' in timing_anomalies_df.columns and not timing_anomalies_df.empty else 0
        
        num_burst_reviewers = len(sliding_window_features_df[sliding_window_features_df['burst_review_count'] > 0]) if not sliding_window_features_df.empty and 'burst_review_count' in sliding_window_features_df.columns else 0
        avg_max_burst_reviews = sliding_window_features_df['max_burst_reviews_in_window'].mean() if not sliding_window_features_df.empty and 'max_burst_reviews_in_window' in sliding_window_features_df.columns else 0

        return {
            "num_reviews_analyzed": len(reviews_df),
            "num_reviewers_with_coordinated_activity_on_products": num_coordinated_reviews,
            "avg_time_diff_between_coordinated_reviews_on_product": avg_time_diff_on_product,
            "avg_rating_variance_on_product_in_coordinated_reviews": avg_rating_variance_on_product,
            "num_template_reviews_identified": num_template_reviews,
            "average_review_burstiness": avg_burstiness,
            "num_reviewers_with_burst_activity": num_burst_reviewers,
            "average_max_reviews_in_burst_window": avg_max_burst_reviews
        }

    def generate_fraud_report(self, comprehensive_results):
        report = "\n--- Comprehensive Fraud Report ---\n"
        report += f"Total Reviews Analyzed: {comprehensive_results['num_reviews_analyzed']}\n"
        report += f"Reviewers with Coordinated Activity on Products: {comprehensive_results['num_reviewers_with_coordinated_activity_on_products']}\n"
        report += f"Average Time Difference (on product) in Coordinated Reviews: {comprehensive_results['avg_time_diff_between_coordinated_reviews_on_product']:.2f} seconds\n"
        report += f"Average Rating Variance (on product) in Coordinated Reviews: {comprehensive_results['avg_rating_variance_on_product_in_coordinated_reviews']:.2f}\n"
        report += f"Template Reviews Identified: {comprehensive_results['num_template_reviews_identified']}\n"
        report += f"Average Review Burstiness Score: {comprehensive_results['average_review_burstiness']:.4f}\n"
        report += f"Reviewers with Burst Activity: {comprehensive_results['num_reviewers_with_burst_activity']}\n"
        report += f"Average Max Reviews in Burst Window: {comprehensive_results['average_max_reviews_in_burst_window']:.2f}\n"
        report += "\n--- Insights ---\n"
        if comprehensive_results['num_reviewers_with_coordinated_activity_on_products'] > 0:
            report += "  - Coordinated activity on specific products suggests targeted fraudulent campaigns.\n"
        if comprehensive_results['num_template_reviews_identified'] > 0:
            report += "  - Identical or highly similar reviews indicate automated generation or template use.\n"
        if comprehensive_results['average_review_burstiness'] > 0.1:
            report += "  - High review burstiness might indicate sudden, inorganic review spikes.\n"
        if comprehensive_results['num_reviewers_with_burst_activity'] > 0:
            report += "  - Significant burst activity suggests potential coordinated efforts or automated review posting within short timeframes.\n"
        report += "\nRecommendations: Investigate flagged reviewers/products for manual review.\n"
        return report

if __name__ == "__main__":
    json_path = '/kaggle/input/amazon-product-review-spam-and-non-spam/Electronics/Electronics.json'

    system = HybridFraudSystem()

    try:
        system.train(json_path)

        if system.cached_df is not None and not system.cached_df.empty:
            sample_reviews_df = system.cached_df.sample(n=min(10, len(system.cached_df)), random_state=42)
            
            sample_reviews_for_inference = sample_reviews_df[['reviewText', 'summary', 'overall', 'reviewerID', 'asin']].to_dict(orient='records')


            print("\n=== CLASSIFYING SAMPLE REVIEWS ===")
            for i, review in enumerate(sample_reviews_for_inference):
                result = system.classify_review(
                    review['reviewText'],
                    review['summary'],
                    review['overall'],
                    review['reviewerID'],
                    review['asin']
                )
                print(f"Review {i+1} ({review['reviewerID']}): {result['classification']} (Confidence: {result['confidence']:.3f})")
                print(f"  Flags: {result['flags']}")
                print(f"  Probabilities: {result['class_probabilities']}")
                print()

            print("\n=== CLASSIFYING SAMPLE SELLERS/PRODUCTS ===")
            unique_asins = system.cached_df['asin'].dropna().unique()
            if len(unique_asins) > 0:
                sample_asin = unique_asins[0]
                print(f"\nClassifying Product (ASIN): {sample_asin}")
                product_classification = system.classify_seller_or_product(sample_asin, id_type='asin')
                print(f"  Classification: {product_classification['classification']} (Confidence: {product_classification['confidence']:.3f})")
                print(f"  Details: {product_classification['details']}")

            unique_reviewer_ids = system.cached_df['reviewerID'].dropna().unique()
            if len(unique_reviewer_ids) > 0:
                sample_reviewer_id = unique_reviewer_ids[0]
                print(f"\nClassifying Reviewer (ReviewerID): {sample_reviewer_id}")
                seller_classification = system.classify_seller_or_product(sample_reviewer_id, id_type='reviewerID')
                print(f"  Classification: {seller_classification['classification']} (Confidence: {seller_classification['confidence']:.3f})")
                print(f"  Details: {seller_classification['details']}")
                
                print(f"\nFinding similar reviewers to {sample_reviewer_id} (Behavioral Fingerprinting):")
                similar_reviewers_result = system.find_similar_reviewers_by_behavior(sample_reviewer_id, num_results=3)
                print(json.dumps(similar_reviewers_result, indent=2))
            

            print("\n=== COMPREHENSIVE FRAUD ANALYSIS (GRAPH-BASED INSIGHTS) ===")
            comprehensive_results = system.comprehensive_fraud_analysis(system.cached_df)
            report = system.generate_fraud_report(comprehensive_results)
            print(report)

            print("=" * 60)
            print("HYBRID TRAINING AND ANALYSIS COMPLETED SUCCESSFULLY!")
            print("The system is now ready to detect fraudulent reviews, and provide insights into potential fraudulent products/sellers.")
            print("Models saved as 'xgboost_fraud_detection_model.joblib', 'scaler.joblib', 'feature_names.json'")
        else:
            print("No data available for sample review classification or comprehensive analysis after training.")

    except FileNotFoundError:
        print(f"Error: Dataset file '{json_path}' not found!")
        print("Please update the json_path variable with the correct path to your dataset.")
    except Exception as e:
        print(f"Error during training or analysis: {str(e)}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support, roc_auc_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("PROPER VALIDATION WITH LEAKAGE MITIGATION")
print("="*80)

def safe_scale_and_model(X_train, X_test, y_train, y_test):
    # Convert y to integer labels starting from 0
    y_train = y_train.astype(int)
    y_test = y_test.astype(int)
    
    # Ensure all column names are strings
    X_train.columns = X_train.columns.astype(str)
    X_test.columns = X_test.columns.astype(str)
    
    non_zero_cols = X_train.columns[X_train.nunique() > 1]
    X_train, X_test = X_train[non_zero_cols], X_test[non_zero_cols]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_scaled = np.nan_to_num(X_train_scaled)
    X_test_scaled = np.nan_to_num(X_test_scaled)
    
    model = xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='logloss', use_label_encoder=False,
        random_state=42, max_depth=5, n_estimators=100
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba) if len(np.unique(y_test)) > 1 else 0.0
    return precision, recall, f1, roc_auc

print("\n[1] GROUPED K-FOLD VALIDATION BY REVIEWER ID")
print("-"*80)

def validate_grouped_by_reviewer(X, y, df_full):
    # Convert y to integer labels
    y = y.astype(int)
    
    gkf = GroupKFold(n_splits=5)
    reviewer_groups = df_full['reviewerID'].values
    fold_metrics = []
    
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=reviewer_groups), 1):
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]

        # Skip fold if training has only one class
        if len(np.unique(y_train_fold)) < 2:
            print(f"\n  Fold {fold} skipped: only one class present in training set")
            continue

        precision, recall, f1, roc_auc = safe_scale_and_model(X_train_fold, X_test_fold, y_train_fold, y_test_fold)
        fold_metrics.append({'fold': fold, 'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': roc_auc})
        print(f"\n  Fold {fold}:")
        print(f"    Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")
        print(f"    Train reviewers: {len(np.unique(reviewer_groups[train_idx]))} | Test reviewers: {len(np.unique(reviewer_groups[test_idx]))}")

    metrics_df = pd.DataFrame(fold_metrics)
    if not metrics_df.empty:
        print(f"\n  AVERAGE METRICS (valid folds only):")
        print(f"    Precision: {metrics_df['precision'].mean():.4f} ± {metrics_df['precision'].std():.4f}")
        print(f"    Recall:    {metrics_df['recall'].mean():.4f} ± {metrics_df['recall'].std():.4f}")
        print(f"    F1-Score:  {metrics_df['f1'].mean():.4f} ± {metrics_df['f1'].std():.4f}")
        print(f"    ROC-AUC:   {metrics_df['roc_auc'].mean():.4f} ± {metrics_df['roc_auc'].std():.4f}")
    else:
        print("No valid folds to compute metrics.")
    return metrics_df

print("\n[2] GROUPED K-FOLD VALIDATION BY PRODUCT ID")
print("-"*80)

def validate_grouped_by_asin(X, y, df_full):
    # Convert y to integer labels
    y = y.astype(int)
    
    gkf = GroupKFold(n_splits=5)
    product_groups = df_full['asin'].values
    fold_metrics = []
    
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=product_groups), 1):
        print(f"\n  Fold {fold}:")
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
        
        precision, recall, f1, roc_auc = safe_scale_and_model(X_train_fold, X_test_fold, y_train_fold, y_test_fold)
        fold_metrics.append({'fold': fold, 'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': roc_auc})
        print(f"    Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")
        print(f"    Train products: {len(np.unique(product_groups[train_idx]))} | Test products: {len(np.unique(product_groups[test_idx]))}")
    
    metrics_df = pd.DataFrame(fold_metrics)
    print(f"\n  AVERAGE METRICS (5-fold GroupKFold by ASIN):")
    print(f"    Precision: {metrics_df['precision'].mean():.4f} ± {metrics_df['precision'].std():.4f}")
    print(f"    Recall:    {metrics_df['recall'].mean():.4f} ± {metrics_df['recall'].std():.4f}")
    print(f"    F1-Score:  {metrics_df['f1'].mean():.4f} ± {metrics_df['f1'].std():.4f}")
    print(f"    ROC-AUC:   {metrics_df['roc_auc'].mean():.4f} ± {metrics_df['roc_auc'].std():.4f}")
    return metrics_df

print("\n[3] TIME-BASED SPLIT (Chronological validation)")
print("-"*80)

def validate_timebased(X, y, df_full):
    # Convert y to integer labels
    y = y.astype(int)
    
    df_temp = X.copy()
    df_temp['timestamp'] = pd.to_datetime(df_full['unixReviewTime'], unit='s')
    df_temp['class'] = y.values
    df_temp = df_temp.sort_values('timestamp').reset_index(drop=True)
    
    split_point = int(0.8 * len(df_temp))
    X_train = df_temp.iloc[:split_point].drop(['timestamp', 'class'], axis=1)
    X_test = df_temp.iloc[split_point:].drop(['timestamp', 'class'], axis=1)
    y_train = df_temp.iloc[:split_point]['class']
    y_test = df_temp.iloc[split_point:]['class']
    
    precision, recall, f1, roc_auc = safe_scale_and_model(X_train, X_test, y_train, y_test)
    print(f"\n  Time-Based Split Results:")
    print(f"    Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")
    
    return {'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': roc_auc}

print("\n[4] ABLATION STUDY: Text-Only vs Text+Advanced Features")
print("-"*80)

def get_text_only_features(X_full):
    # Ensure column names are strings
    X_full.columns = X_full.columns.astype(str)
    
    text_cols = [col for col in X_full.columns if col.startswith('text_feature_') or col.startswith('summary_feature_')]
    return X_full[text_cols] if text_cols else X_full.iloc[:, :100]

def validate_ablation(X_full, X_text_only, y):
    # Convert y to integer labels
    y = y.astype(int)
    
    X_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42, stratify=y)
    X_train_text, X_test_text, _, _ = train_test_split(X_text_only, y, test_size=0.2, random_state=42, stratify=y)
    
    results = {}
    precision_text, recall_text, f1_text, roc_auc_text = safe_scale_and_model(X_train_text, X_test_text, y_train, y_test)
    results['text_only'] = {'precision': precision_text, 'recall': recall_text, 'f1': f1_text, 'roc_auc': roc_auc_text}
    
    precision_full, recall_full, f1_full, roc_auc_full = safe_scale_and_model(X_train_full, X_test_full, y_train, y_test)
    results['full'] = {'precision': precision_full, 'recall': recall_full, 'f1': f1_full, 'roc_auc': roc_auc_full}
    
    print(f"\n  ABLATION COMPARISON:")
    print(f"    Text-Only:    Precision: {precision_text:.4f} | Recall: {recall_text:.4f} | F1: {f1_text:.4f} | ROC-AUC: {roc_auc_text:.4f}")
    print(f"    Full Features: Precision: {precision_full:.4f} | Recall: {recall_full:.4f} | F1: {f1_full:.4f} | ROC-AUC: {roc_auc_full:.4f}")
    print(f"    Feature importance (F1 improvement): {(f1_full - f1_text):.4f}")
    print(f"    → Advanced features contribute: {((f1_full - f1_text) / f1_text * 100 if f1_text > 0 else 0):.2f}% F1 improvement")
    
    return results

# --- Prepare numeric features ---
print("Preparing validation data...")
X_validation = system.full_feature_df.select_dtypes(include=[np.number]).drop(['class'], axis=1, errors='ignore')
y_validation = system.full_feature_df['class']
df_validation = system.full_feature_df.copy()

# Ensure all column names are strings
X_validation.columns = X_validation.columns.astype(str)

print(f"Validation data shape: {X_validation.shape}")
print(f"Class distribution: {y_validation.value_counts()}")

# Run validation
print("\nRunning validation experiments...")
grouped_metrics = validate_grouped_by_reviewer(X_validation, y_validation, df_validation)
asin_metrics = validate_grouped_by_asin(X_validation, y_validation, df_validation)
timebased_metrics = validate_timebased(X_validation, y_validation, df_validation)

X_text_only = get_text_only_features(X_validation)
ablation_results = validate_ablation(X_validation, X_text_only, y_validation)

# --- Summary ---
print("\n" + "="*80)
print("VALIDATION SUMMARY")
print("="*80)

summary_data = []
if not grouped_metrics.empty:
    summary_data.append(['Grouped by ReviewerID', grouped_metrics['precision'].mean(), grouped_metrics['recall'].mean(), 
                        grouped_metrics['f1'].mean(), grouped_metrics['roc_auc'].mean()])

if not asin_metrics.empty:
    summary_data.append(['Grouped by ASIN', asin_metrics['precision'].mean(), asin_metrics['recall'].mean(), 
                        asin_metrics['f1'].mean(), asin_metrics['roc_auc'].mean()])

summary_data.append(['Time-Based Split', timebased_metrics['precision'], timebased_metrics['recall'], 
                    timebased_metrics['f1'], timebased_metrics['roc_auc']])
summary_data.append(['Text-Only (Ablation)', ablation_results['text_only']['precision'], 
                    ablation_results['text_only']['recall'], ablation_results['text_only']['f1'], 
                    ablation_results['text_only']['roc_auc']])
summary_data.append(['Full Model (Ablation)', ablation_results['full']['precision'], 
                    ablation_results['full']['recall'], ablation_results['full']['f1'], 
                    ablation_results['full']['roc_auc']])

summary = pd.DataFrame(summary_data, 
                      columns=['Validation Method', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

print("\n" + summary.to_string(index=False))

print(f"\n{'='*80}")
print("KEY FINDINGS:")
print(f"{'='*80}")
if ablation_results['text_only']['f1'] > 0:
    improvement = ((ablation_results['full']['f1'] - ablation_results['text_only']['f1']) / ablation_results['text_only']['f1'] * 100)
    print(f"✓ Advanced features improve F1 by: {improvement:.2f}%")
else:
    print("✓ Advanced features provide additional detection capabilities")
print("✓ Proper grouped validation prevents data leakage")
print("✓ Multiple validation strategies provide robust performance estimation")

In [ ]:
import nbformat, json, os, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb
import joblib

# Fix 1: Check for the correct attribute name (camelCase vs lowercase)
if 'X_validation' not in locals() or 'df_validation' not in locals():
    if hasattr(system, 'full_feature_df') and system.full_feature_df is not None:
        df_validation = system.full_feature_df.copy()
    elif hasattr(system, 'fullfeaturedf') and system.fullfeaturedf is not None:
        df_validation = system.fullfeaturedf.copy()
    else:
        raise RuntimeError("system.full_feature_df not found. Run training first.")
    
    # Fix 2: Ensure we're dropping the correct columns
    X_validation = df_validation.drop(
        ['class', 'reviewerID', 'reviewText', 'summary', 'asin',
         'unixReviewTime', 'reviewTime', 'overall', 'processed_text_for_hash'],
        axis=1, errors='ignore'
    )
    # Fix 3: Convert class labels to integers
    y_validation = df_validation['class'].astype(int)

group_col = 'reviewerID'
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)
groups = df_validation[group_col].values
raw_text = df_validation.get('reviewText', None)

fold_results = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(X_validation, y_validation, groups=groups), 1):
    X_train_full, X_test_full = X_validation.iloc[train_idx].reset_index(drop=True), X_validation.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y_validation.iloc[train_idx].reset_index(drop=True), y_validation.iloc[test_idx].reset_index(drop=True)
    
    # Fix 4: Ensure integer labels
    y_train = y_train.astype(int)
    y_test = y_test.astype(int)
    
    if raw_text is not None:
        vect = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
        X_train_text = vect.fit_transform(raw_text.iloc[train_idx]).toarray()
        X_test_text = vect.transform(raw_text.iloc[test_idx]).toarray()
        # Fix 5: Create proper column names for TF-IDF features
        X_train_text = pd.DataFrame(X_train_text, columns=[f'tfidf_{i}' for i in range(X_train_text.shape[1])])
        X_test_text = pd.DataFrame(X_test_text, columns=[f'tfidf_{i}' for i in range(X_test_text.shape[1])])
    else:
        text_cols = [c for c in X_validation.columns if c.startswith('text_feature_') or c.startswith('summary_feature_')]
        if not text_cols:
            raise RuntimeError("No raw text or precomputed text features found.")
        X_train_text = X_validation.iloc[train_idx][text_cols].reset_index(drop=True)
        X_test_text = X_validation.iloc[test_idx][text_cols].reset_index(drop=True)
    
    # Fix 6: Ensure ALL column names are strings before concatenation
    X_train_full.columns = X_train_full.columns.astype(str)
    X_test_full.columns = X_test_full.columns.astype(str)
    X_train_text.columns = X_train_text.columns.astype(str)
    X_test_text.columns = X_test_text.columns.astype(str)
    
    X_train = pd.concat([X_train_text, X_train_full], axis=1)
    X_test = pd.concat([X_test_text, X_test_full], axis=1)
    
    # Fix 7: Ensure the concatenated DataFrames also have string columns
    X_train.columns = X_train.columns.astype(str)
    X_test.columns = X_test.columns.astype(str)
    
    # Fix 8: Handle case where inner split might have only one class
    if len(np.unique(y_train)) < 2:
        print(f"Fold {fold} skipped: only one class in training data after split")
        continue
        
    inner_train_idx, inner_val_idx = train_test_split(
        np.arange(len(X_train)), test_size=0.2, random_state=42, stratify=y_train
    )
    X_tr, X_val = X_train.iloc[inner_train_idx], X_train.iloc[inner_val_idx]
    y_tr, y_val = y_train.iloc[inner_train_idx], y_train.iloc[inner_val_idx]
    
    # Fix 9: Ensure integer labels for inner splits
    y_tr = y_tr.astype(int)
    y_val = y_val.astype(int)
    
    # Fix 10: Convert to numpy arrays to avoid column name issues entirely
    # This is the key fix - sklearn's StandardScaler has issues with mixed column name types
    X_tr_values = X_tr.values if hasattr(X_tr, 'values') else X_tr
    X_val_values = X_val.values if hasattr(X_val, 'values') else X_val  
    X_test_values = X_test.values if hasattr(X_test, 'values') else X_test
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr_values)
    X_val_s = scaler.transform(X_val_values)
    X_test_s = scaler.transform(X_test_values)
    
    clf = xgb.XGBClassifier(
        objective='binary:logistic',
        use_label_encoder=False,
        eval_metric='auc',
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=1.0,
        reg_lambda=2.0,
        random_state=42,
        n_jobs=-1
    )
    
    try:
        clf.fit(
            X_tr_s, y_tr,
            early_stopping_rounds=50,
            eval_set=[(X_val_s, y_val)],
            verbose=False
        )
        
        y_pred = clf.predict(X_test_s)
        y_proba = clf.predict_proba(X_test_s)[:,1]
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
        roc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.0
        
        fold_results.append({
            'fold': fold,
            'precision': precision,
            'recall': recall, 
            'f1': f1,
            'roc_auc': roc, 
            'n_tree_used': getattr(clf, 'best_iteration', None) or clf.n_estimators
        })
        
        joblib.dump(clf, f'xgb_fold_{fold}.joblib')
        print(f"Fold {fold}: precision={precision:.4f} recall={recall:.4f} f1={f1:.4f} roc_auc={roc:.4f} trees_used={fold_results[-1]['n_tree_used']}")
        
    except Exception as e:
        print(f"Fold {fold} failed: {str(e)}")
        continue

if fold_results:
    res_df = pd.DataFrame(fold_results)
    print("\nSummary:")
    print(res_df.describe().loc[['mean','std']])
    
    # Also show individual fold results
    print("\nDetailed Results:")
    for result in fold_results:
        print(f"Fold {result['fold']}: F1={result['f1']:.4f}, AUC={result['roc_auc']:.4f}")
else:
    print("No successful folds to summarize.")